### `Ideia do notebook (v2 — melhorias aplicadas):`

Usar RoBERTa (blair) para prever peso, altura, largura e comprimento a partir de:
- **título + categoria + descrição** (em vez de só o título)
- **múltiplas categorias** (em vez de filtrar só Beauty & Personal Care)
- **escala logarítmica** para os alvos (em vez de StandardScaler linear)
- **fine-tuning progressivo** (warm-up congelado → descongelar últimas 4 camadas)
- **lr=1e-4 + warmup linear + decaimento cosseno** (em vez de lr=1e-3 fixo)

**Melhorias implementadas:**
1. 🔴 Dataset ampliado: múltiplas categorias + categoria como feature de entrada
2. 🔴 Input enriquecido: `título [SEP] categoria [SEP] descrição[:300]` + extração regex de dimensões
3. 🟠 Fine-tuning progressivo: 3 épocas congelado → descongelar últimas 4 camadas
4. 🟠 Alvos em escala logarítmica: `log1p` nos targets, `expm1` na inversão
5. 🟠 Otimização: `lr=1e-4`, `warmup linear` + `CosineAnnealingLR`

### Setup inicial

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando o device: {device}")

### Carregando modelo e tokenizer

In [ ]:
MODEL_NAME = "hyp1231/blair-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

### Melhoria 1 & 2 — Dataset ampliado + Input enriquecido

**Antes:** filtrava só `Beauty & Personal Care > Skin Care`, resultando em ~600 exemplos, e usava apenas o título.

**Agora:**
- Sem filtro de categoria → dataset completo (~40k exemplos)
- Input = `título [SEP] categoria_folha [SEP] descrição[:300]`
- Extração regex de dimensões explícitas no texto ("7 x 7 x 1.3cm", "20' H x 26' W")

In [ ]:
caminho_csv = "../data/cubagem_40k_amazon.csv"
df = pd.read_csv(caminho_csv)

# --- Melhoria 1: SEM filtro de categoria — usa dataset completo ---
# CATEGORIA_ALVO removida: treinamos em todas as categorias.
# A categoria é passada como feature de entrada para o modelo aprender
# que "compact mirror" é pequeno e "water bottle" é alto.

colunas_alvo = ['length_cm', 'width_cm', 'height_cm', 'weight_g']

# Extrai a categoria folha (último nível) para usar como feature de contexto
def categoria_folha(cat_str):
    """Retorna o último nível da hierarquia de categorias."""
    if pd.isna(cat_str):
        return ""
    partes = [p.strip() for p in str(cat_str).split(">")]
    return partes[-1] if partes else ""

df['categoria_folha'] = df['categories'].apply(categoria_folha)

# --- Melhoria 2: Extração regex de dimensões explícitas ---
# Captura padrões como "7 x 7 x 1.3 cm", "20" H x 26" W", "5.5 x 3.2 x 1.1 inches"
DIM_REGEX = re.compile(
    r"(\d+\.?\d*)\s*[xX×]\s*(\d+\.?\d*)\s*[xX×]?\s*(\d+\.?\d*)?\s*(cm|inch|inches|\"|\')",
    re.IGNORECASE
)

def extrair_dims_texto(texto):
    """Extrai menções de dimensões do texto e retorna como string normalizada."""
    if pd.isna(texto):
        return ""
    matches = DIM_REGEX.findall(str(texto))
    if not matches:
        return ""
    # Formata a primeira ocorrência encontrada
    d1, d2, d3, unit = matches[0]
    dims = [d for d in [d1, d2, d3] if d]
    return f"dimensions: {' x '.join(dims)} {unit}"

df['dims_extraidas'] = df['description'].apply(extrair_dims_texto)

# --- Monta o texto de entrada: título + categoria + dims extraídas + descrição truncada ---
def montar_texto_entrada(row):
    titulo = str(row['title']) if pd.notna(row['title']) else ""
    categoria = str(row['categoria_folha'])
    dims = str(row['dims_extraidas'])  # dimensões extraídas por regex (pode ser vazio)
    descricao = str(row['description'])[:300] if pd.notna(row['description']) else ""
    
    partes = [titulo, categoria]
    if dims:
        partes.append(dims)  # adiciona dims explícitas se encontradas
    partes.append(descricao)
    
    return " [SEP] ".join(partes)

df['texto_entrada'] = df.apply(montar_texto_entrada, axis=1)

df_train = df[df['split'] == 'train'].dropna(subset=['texto_entrada'] + colunas_alvo)
df_val   = df[df['split'] == 'val'].dropna(subset=['texto_entrada'] + colunas_alvo)

print(f"Total de produtos de treino: {len(df_train)}")
print(f"Total de produtos de validação: {len(df_val)}")
print(f"\nExemplo de texto de entrada:")
print(df_train['texto_entrada'].iloc[0][:200])

### Melhoria 4 — Alvos em escala logarítmica

**Antes:** `StandardScaler` em escala linear. Com distribuições assimétricas (itens de 60g a 5kg), o scaler não resolve a cauda pesada.

**Agora:** `log1p` nos targets antes de qualquer normalização. O modelo prevê em log-space e a inversão é `expm1`. O Huber loss converge muito mais rápido em escala log.

In [ ]:
# ---------------------------------------------------------
# MELHORIA 4: Alvos em escala logarítmica
# O CSV já tem log_volume e log_weight calculados — seguimos
# a mesma lógica para os 4 alvos individualmente.
# log1p(x) = log(1 + x) é seguro para valores próximos de 0.
# ---------------------------------------------------------
textos_train  = df_train['texto_entrada'].values
targets_train = df_train[colunas_alvo].values.astype(np.float32)

textos_val    = df_val['texto_entrada'].values
targets_val   = df_val[colunas_alvo].values.astype(np.float32)

# Aplicar log1p — distribui os valores numa escala mais simétrica
targets_train_log = np.log1p(targets_train)
targets_val_log   = np.log1p(targets_val)

# Normalizar pela média/std do treino APÓS a transformação log
log_mean = targets_train_log.mean(axis=0)
log_std  = targets_train_log.std(axis=0) + 1e-8  # evita divisão por zero

targets_train_norm = (targets_train_log - log_mean) / log_std
targets_val_norm   = (targets_val_log   - log_mean) / log_std

print(f"Média log-normalizada (treino): {log_mean.round(3)}")
print(f"Std  log-normalizada (treino): {log_std.round(3)}")

def inverter_predicao(arr_norm):
    """Reverte normalização log → escala original (cm, g)."""
    arr_log = arr_norm * log_std + log_mean
    return np.expm1(arr_log)


# ---------------------------------------------------------
# DATASET E DATALOADERS
# max_length aumentado para 256 para acomodar o texto mais rico
# ---------------------------------------------------------
class CubagemDataset(Dataset):
    def __init__(self, textos, targets, tokenizer, max_len=256):
        self.textos   = textos
        self.targets  = targets
        self.tokenizer = tokenizer
        self.max_len  = max_len

    def __len__(self):
        return len(self.textos)

    def __getitem__(self, idx):
        texto  = str(self.textos[idx])
        target = self.targets[idx]

        encoding = self.tokenizer(
            texto,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'targets':        torch.tensor(target, dtype=torch.float32),
            'texto_original': texto
        }

dataset_train = CubagemDataset(textos_train, targets_train_norm, tokenizer)
dataloader    = DataLoader(dataset_train, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

dataset_val   = CubagemDataset(textos_val, targets_val_norm, tokenizer)
val_dataloader = DataLoader(dataset_val, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

### Melhoria 3 — Arquitetura com Fine-tuning Progressivo

**Antes:** base model 100% congelado em todas as épocas.

**Agora:**
- **Fase 1 (warm-up, épocas 1–3):** base congelado, só o MLP aprende. Evita destruir os embeddings cedo.
- **Fase 2 (fine-tune, épocas 4+):** descongelamos as últimas 4 camadas do encoder com `lr=1e-5`. O MLP continua com `lr=1e-4`.

In [ ]:
class ModeloCubagem(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(model_name)

        # FASE 1: base congelado por padrão
        for param in self.base_model.parameters():
            param.requires_grad = False

        hidden_size = self.base_model.config.hidden_size  # 768

        # MLP mais profundo para aproveitar o input enriquecido
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 4)  # 4 outputs: length_cm, width_cm, height_cm, weight_g
        )

    def descongelar_ultimas_camadas(self, n_camadas=4):
        """Descongela as últimas n camadas do encoder RoBERTa."""
        # RoBERTa base tem 12 camadas (encoder.layer[0..11])
        total_layers = len(self.base_model.encoder.layer)
        for i, layer in enumerate(self.base_model.encoder.layer):
            if i >= total_layers - n_camadas:
                for param in layer.parameters():
                    param.requires_grad = True
        # Também descongela o pooler
        for param in self.base_model.pooler.parameters():
            param.requires_grad = True
        print(f"Últimas {n_camadas} camadas do encoder descongeladas.")

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # token [CLS]
        return self.mlp(cls_embedding)


modelo = ModeloCubagem(MODEL_NAME).to(device)
print(f"Parâmetros treináveis (fase warm-up): "
      f"{sum(p.numel() for p in modelo.parameters() if p.requires_grad):,}")

### Melhoria 5 — Otimizador: lr=1e-4 + Warmup Linear + CosineAnnealingLR

**Antes:** `Adam(lr=1e-3)` fixo, sem schedule.

**Agora:**
- `lr=1e-4` para o MLP (10× menor que antes)
- `lr=1e-5` para as camadas do encoder na Fase 2
- Warmup linear nos primeiros 10% dos steps
- `CosineAnnealingLR` para decaimento suave até o final do treino

In [ ]:
# ---------------------------------------------------------
# CONFIGURAÇÕES DE TREINO
# ---------------------------------------------------------
EPOCHS_WARMUP    = 3   # Fase 1: só MLP, base congelado
EPOCHS_FINETUNE  = 5   # Fase 2: MLP + últimas 4 camadas do encoder
EPOCHS_TOTAL     = EPOCHS_WARMUP + EPOCHS_FINETUNE

LR_MLP           = 1e-4   # lr para o MLP (Melhoria 5)
LR_ENCODER       = 1e-5   # lr para as camadas descongeladas (Fase 2)
N_CAMADAS_UNFREEZE = 4    # Quantas camadas do encoder descongelar (Melhoria 3)
WARMUP_RATIO     = 0.1    # 10% dos steps totais como warmup linear

criterion = nn.HuberLoss()  # robusto a outliers

# Otimizador inicial: só parâmetros do MLP
optimizer = optim.AdamW(modelo.mlp.parameters(), lr=LR_MLP, weight_decay=1e-2)

total_steps   = len(dataloader) * EPOCHS_TOTAL
warmup_steps  = int(total_steps * WARMUP_RATIO)

# Warmup linear seguido de decaimento cosseno
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f"Total de steps: {total_steps} | Warmup steps: {warmup_steps}")
print(f"Épocas warm-up (base congelado): {EPOCHS_WARMUP}")
print(f"Épocas fine-tune (últimas {N_CAMADAS_UNFREEZE} camadas): {EPOCHS_FINETUNE}")
print()

melhor_val_loss = float('inf')

for epoch in range(EPOCHS_TOTAL):

    # ==========================================
    # MELHORIA 3: Transição para Fase 2 (Fine-tune)
    # ==========================================
    if epoch == EPOCHS_WARMUP:
        modelo.descongelar_ultimas_camadas(N_CAMADAS_UNFREEZE)

        # Recria o otimizador com dois grupos de parâmetros e lrs distintos
        encoder_params = [p for p in modelo.base_model.parameters() if p.requires_grad]
        optimizer = optim.AdamW([
            {'params': modelo.mlp.parameters(),  'lr': LR_MLP},
            {'params': encoder_params,            'lr': LR_ENCODER},
        ], weight_decay=1e-2)

        # Recria o scheduler para os steps restantes
        steps_restantes = len(dataloader) * EPOCHS_FINETUNE
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=0,
            num_training_steps=steps_restantes
        )
        print(f"[Época {epoch+1}] → Iniciando FASE 2 (fine-tune encoder)")

    fase = "warm-up" if epoch < EPOCHS_WARMUP else "fine-tune"

    # ==========================================
    # TREINAMENTO
    # ==========================================
    modelo.train()
    if epoch < EPOCHS_WARMUP:
        modelo.base_model.eval()  # base congelado na Fase 1

    train_loss = 0

    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        targets_batch  = batch['targets'].to(device)

        optimizer.zero_grad()
        predicoes = modelo(input_ids, attention_mask)
        loss = criterion(predicoes, targets_batch)
        loss.backward()

        # Gradient clipping para estabilidade
        nn.utils.clip_grad_norm_(modelo.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()  # atualiza lr a cada step
        train_loss += loss.item()

    avg_train_loss = train_loss / len(dataloader)

    # ==========================================
    # VALIDAÇÃO
    # ==========================================
    modelo.eval()
    val_loss = 0
    exemplos_capturados = False

    with torch.no_grad():
        for batch in val_dataloader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets_batch  = batch['targets'].to(device)

            predicoes = modelo(input_ids, attention_mask)
            loss = criterion(predicoes, targets_batch)
            val_loss += loss.item()

            if not exemplos_capturados:
                preds_np   = predicoes.cpu().numpy()
                targets_np = targets_batch.cpu().numpy()

                # Melhoria 4: inversão via expm1 (em vez de scaler.inverse_transform)
                preds_reais   = inverter_predicao(preds_np)
                targets_reais = inverter_predicao(targets_np)
                exemplos_capturados = True

    avg_val_loss = val_loss / len(val_dataloader)
    lr_atual = optimizer.param_groups[0]['lr']

    # Checkpoint: salva o melhor modelo
    if avg_val_loss < melhor_val_loss:
        melhor_val_loss = avg_val_loss
        torch.save(modelo.state_dict(), "melhor_modelo_cubagem.pt")
        ckpt_str = " ← melhor até agora"
    else:
        ckpt_str = ""

    # ==========================================
    # EXEMPLOS DE VALIDAÇÃO A CADA ÉPOCA
    # ==========================================
    print(f"\n[Época {epoch+1}/{EPOCHS_TOTAL} | {fase}] "
          f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
          f"| lr: {lr_atual:.2e}{ckpt_str}")
    print("-" * 70)
    print("Visualizando predições (Validação):")

    for i in range(min(2, len(preds_reais))):
        print(f"  > Exemplo {i+1}:")
        print(f"    Real -> Comp: {targets_reais[i][0]:.1f}cm | Larg: {targets_reais[i][1]:.1f}cm "
              f"| Alt: {targets_reais[i][2]:.1f}cm | Peso: {targets_reais[i][3]:.1f}g")
        print(f"    Pred -> Comp: {preds_reais[i][0]:.1f}cm | Larg: {preds_reais[i][1]:.1f}cm "
              f"| Alt: {preds_reais[i][2]:.1f}cm | Peso: {preds_reais[i][3]:.1f}g")

    print("=" * 70)

print(f"\nTreinamento concluído. Melhor val loss: {melhor_val_loss:.4f}")
print("Modelo salvo em: melhor_modelo_cubagem.pt")

### Inferência — usando o melhor checkpoint

In [ ]:
# Carrega o melhor modelo salvo durante o treino
modelo.load_state_dict(torch.load("melhor_modelo_cubagem.pt", map_location=device))
modelo.eval()

def prever_dimensoes(titulo, categoria="", descricao=""):
    """
    Prevê [length_cm, width_cm, height_cm, weight_g] a partir de metadados do produto.
    """
    dims = extrair_dims_texto(descricao)
    partes = [titulo, categoria]
    if dims:
        partes.append(dims)
    partes.append(descricao[:300])
    texto = " [SEP] ".join(partes)

    enc = tokenizer(texto, max_length=256, padding='max_length',
                    truncation=True, return_tensors='pt')
    with torch.no_grad():
        pred_norm = modelo(
            enc['input_ids'].to(device),
            enc['attention_mask'].to(device)
        ).cpu().numpy()

    resultado = inverter_predicao(pred_norm)[0]
    return dict(zip(colunas_alvo, resultado.tolist()))

# Exemplo
exemplo = prever_dimensoes(
    titulo="Alba Botanica Very Emollient Body Lotion Maximum Dry Skin 32 oz",
    categoria="Moisturizers",
    descricao="Alba Botanica Very Emollient Body Lotion Maximum Dry Skin 32 oz"
)
print("Predição de exemplo:")
for k, v in exemplo.items():
    print(f"  {k}: {v:.2f}")